[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_56_Phase6_OSS_Kickoff.ipynb)

# Lesson 56 — Phase 6 Kickoff: From Learner to OSS Builder 🚀

**Date:** 2026-06-28  
**Phase:** 6 of 6 — Build, Ship & Grow  
**Track:** Real Open-Source AI Project

---

## You made it. 55 lessons. 5 phases. Now it counts.

You've learned:
- **Phase 1:** LLM fundamentals → prompt engineering → tools → ReAct agents
- **Phase 2:** Memory, multi-agent systems, A2A protocols, reliability patterns
- **Phase 3:** Fine-tuning, QLoRA, DPO, distillation, model merging
- **Phase 4:** Multimodal — voice, image, document AI
- **Phase 5:** Production — durable execution, GPU autoscaling, OTel, eval at scale, PyPI ship

That's the knowledge. Phase 6 is different.  
**Phase 6 is about building something real that the world can use.**

> "The only way to prove you're an AI engineer is to ship one."

---

## Phase 6 Goal

By the end of Phase 6 you will have:
1. A published, starred open-source AI project on GitHub
2. A working demo you can show in interviews
3. Real users giving you feedback
4. Contributions back to the broader ecosystem

Let's pick the project and start building today.


## Phase 6 Roadmap (Lessons 56–65)

| Lesson | Title | What You Build |
|--------|-------|----------------|
| **L56** | Phase 6 Kickoff ← *you are here* | Project selection + MVP pipeline |
| L57 | paper-distiller: Core Pipeline | arXiv fetch → Claude extraction → structured output |
| L58 | Evals on Real Papers | Golden set, recall@k, LLM judge for accuracy |
| L59 | OSS Growth Playbook | README, HuggingFace Space, demo video |
| L60 | CLI + PyPI Release | `pip install paper-distiller`, rich CLI |
| L61 | Web API + Streaming | FastAPI, SSE streaming, CORS |
| L62 | Agent Benchmark (agent-bench) | Evaluating agents across tasks systematically |
| L63 | Integrating External Data | Semantic Scholar, Zotero, Obsidian export |
| L64 | Safety & Responsible AI Patterns | Hallucination detection, citation grounding |
| L65 | **Capstone: Public Launch Day** | GitHub release, blog post, HN submission |

**Today (L56):** Choose the project, design the architecture, build the working MVP pipeline end-to-end.


## Choosing the Right Project

At the end of Phase 5 (Lesson 55), three open-source ideas were proposed:

| Project | What it does |
|---------|-------------|
| **paper-distiller** | Takes an arXiv paper → extracts methods, results, code examples, practitioner TL;DR |
| **agent-bench** | Framework to benchmark AI agents across diverse tasks with reproducible metrics |
| **auto-researcher** | The multi-agent research pipeline built in L51–L55, packaged for PyPI |

Let's score them objectively before picking one.


In [ ]:
import textwrap

# Decision matrix: score each criterion 1-5 for each project
criteria = {
    "Demo-ability (can you show it in 30 sec?)": [5, 3, 4],
    "Novelty (does it fill a clear gap?)":        [5, 4, 3],
    "Completion speed (MVP in <1 week?)":         [5, 3, 3],
    "Real user demand":                           [5, 4, 3],
    "Builds on what you already have":            [4, 2, 5],
    "Portfolio signal (impressive to hiring mgr)": [5, 4, 4],
}
weights = {
    "Demo-ability (can you show it in 30 sec?)":  0.20,
    "Novelty (does it fill a clear gap?)":         0.20,
    "Completion speed (MVP in <1 week?)":          0.15,
    "Real user demand":                            0.20,
    "Builds on what you already have":             0.10,
    "Portfolio signal (impressive to hiring mgr)": 0.15,
}
projects = ["paper-distiller", "agent-bench", "auto-researcher"]

print("=" * 72)
print(f"{'CRITERION':<44}  {'W':>4}  {'PD':>5}  {'AB':>5}  {'AR':>5}")
print("=" * 72)

totals = [0.0, 0.0, 0.0]
for criterion, scores in criteria.items():
    w = weights[criterion]
    label = textwrap.shorten(criterion, width=44, placeholder="…")
    print(f"{label:<44}  {w:>4.2f}  {scores[0]:>5}  {scores[1]:>5}  {scores[2]:>5}")
    for i, s in enumerate(scores):
        totals[i] += s * w

print("-" * 72)
print(f"{'WEIGHTED TOTAL':<44}  {'':>4}", end="")
for t in totals:
    print(f"  {t:>5.2f}", end="")
print()
print("=" * 72)

winner_idx = totals.index(max(totals))
print(f"\n🏆 Winner: {projects[winner_idx]} (score: {totals[winner_idx]:.2f})")


## Why paper-distiller?

1. **Immediate demo**: Give it any arXiv URL → get a structured breakdown + code snippet in seconds. Anyone can see the value instantly.

2. **Clear gap**: Reading papers is painful. Researchers, engineers, students all want: *"what do I need to know and how do I use it?"* — not 30 pages of LaTeX.

3. **Showcases your stack**: arXiv API → Claude extraction → structured Pydantic models → LLM-judged eval → CLI. Every lesson you learned gets used.

4. **Reusable signal**: You can submit it to HuggingFace Spaces, show it in a blog post, and reference it in GitHub Copilot/Anthropic job applications.

5. **Compound opportunity**: paper-distiller can power agent-bench (distill agent papers → extract benchmarks) and auto-researcher (use it as a retrieval layer). They compose.

---

## Architecture

```
Input: arXiv URL or paper ID
         │
         ▼
┌─────────────────────────────────────────────────────────┐
│  FetchLayer                                             │
│  • arXiv API → metadata (title, authors, abstract)      │
│  • PDF download → pdfplumber text extraction            │
│  • Section detection (intro/methods/results/conclusion) │
└─────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────┐
│  ExtractLayer  (Claude Sonnet)                          │
│  • Core contribution (1 sentence)                       │
│  • Key method (plain English)                           │
│  • Benchmark results table                              │
│  • Prerequisites (what you need to understand first)    │
│  • Limitations (what the paper doesn't cover)           │
└─────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────┐
│  CodeLayer  (Claude Haiku)                              │
│  • Minimal runnable Python implementation of core idea  │
│  • Uses only stdlib + numpy/torch                       │
└─────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────┐
│  Output: PaperDigest (Pydantic)                         │
│  • Markdown summary                                     │
│  • JSON structured data                                 │
│  • Code snippet                                         │
└─────────────────────────────────────────────────────────┘
```


In [ ]:
# Setup — install all deps
!pip install anthropic pdfplumber pydantic requests rich nest_asyncio -q

import os, json, re, textwrap, asyncio
import requests
import pdfplumber
import io
import anthropic
from pydantic import BaseModel, Field
from typing import Optional
import nest_asyncio
nest_asyncio.apply()

# Load API key (Colab Secrets or env var)
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = os.environ.get("ANTHROPIC_API_KEY", "sk-ant-YOUR-KEY-HERE")

client = anthropic.Anthropic(api_key=api_key)
print("✅ Setup complete")


In [ ]:
# ── FetchLayer ──────────────────────────────────────────────────────────────
import xml.etree.ElementTree as ET

ARXIV_API = "http://export.arxiv.org/api/query"
ARXIV_PDF  = "https://arxiv.org/pdf/{paper_id}"

def parse_arxiv_id(url_or_id: str) -> str:
    """Extract clean arxiv ID from any format: URL, abs, pdf, or bare ID."""
    # Strip trailing .pdf
    url_or_id = url_or_id.strip().rstrip("/").replace(".pdf", "")
    # Patterns: 2301.12345 or 2301.12345v2
    m = re.search(r"(\d{4}\.\d{4,5}(?:v\d+)?)", url_or_id)
    if m:
        return m.group(1)
    raise ValueError(f"Cannot parse arXiv ID from: {url_or_id!r}")

def fetch_metadata(paper_id: str) -> dict:
    """Fetch title, authors, abstract from arXiv API."""
    resp = requests.get(ARXIV_API, params={"id_list": paper_id, "max_results": 1}, timeout=15)
    resp.raise_for_status()
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    root = ET.fromstring(resp.text)
    entry = root.find("atom:entry", ns)
    if entry is None:
        raise ValueError(f"No entry found for paper_id={paper_id}")
    title    = entry.findtext("atom:title", namespaces=ns, default="").strip().replace("\n", " ")
    abstract = entry.findtext("atom:summary", namespaces=ns, default="").strip().replace("\n", " ")
    authors  = [a.findtext("atom:name", namespaces=ns) for a in entry.findall("atom:author", ns)]
    year_str = entry.findtext("atom:published", namespaces=ns, default="")[:4]
    return {"paper_id": paper_id, "title": title, "abstract": abstract,
            "authors": authors, "year": year_str}

def fetch_pdf_text(paper_id: str, max_chars: int = 18_000) -> str:
    """Download PDF and extract text (first max_chars characters)."""
    url = ARXIV_PDF.format(paper_id=paper_id)
    resp = requests.get(url, timeout=30, headers={"User-Agent": "paper-distiller/0.1"})
    resp.raise_for_status()
    with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
        pages_text = [page.extract_text() or "" for page in pdf.pages[:20]]
    full_text = "\n\n".join(pages_text)
    return full_text[:max_chars]

# ── Quick smoke test ──────────────────────────────────────────────────────────
# We use the "Attention Is All You Need" paper as a well-known, stable test case
DEMO_PAPER = "1706.03762"

meta = fetch_metadata(DEMO_PAPER)
print(f"📄 Title:   {meta['title']}")
print(f"👥 Authors: {', '.join(meta['authors'][:3])} et al.")
print(f"📅 Year:    {meta['year']}")
print(f"📝 Abstract (first 300 chars):\n   {meta['abstract'][:300]}...")


In [ ]:
# ── PaperDigest Pydantic Model ───────────────────────────────────────────────

class BenchmarkResult(BaseModel):
    dataset: str = Field(description="Dataset or benchmark name")
    metric:  str = Field(description="Metric name (BLEU, accuracy, F1, perplexity…)")
    value:   str = Field(description="The reported value as a string (e.g. '94.8%')")
    comparison: Optional[str] = Field(None, description="What it's compared against (prior SOTA, baseline)")

class PaperDigest(BaseModel):
    title:          str   = Field(description="Paper title")
    one_liner:      str   = Field(description="Core contribution in ONE sentence (<25 words), plain English, no jargon")
    method_summary: str   = Field(description="How the method works in 3-5 plain sentences. No equations. No citations.")
    key_results:    list[BenchmarkResult] = Field(description="Up to 5 key benchmark or quantitative results")
    prerequisites:  list[str] = Field(description="2-4 concepts the reader must understand first")
    limitations:    list[str] = Field(description="2-3 honest limitations the paper acknowledges or that are obvious")
    practitioner_tldr: str = Field(description="What a software engineer should DO with this paper in 2-3 sentences")
    tags:           list[str] = Field(description="3-5 topic tags like ['transformers', 'attention', 'NLP', 'seq2seq']")

print("✅ PaperDigest model defined")
print("   Fields:", list(PaperDigest.model_fields.keys()))


In [ ]:
# ── ExtractLayer ─────────────────────────────────────────────────────────────

EXTRACT_TOOL = {
    "name": "extract_paper_digest",
    "description": "Extract a structured digest from a research paper.",
    "input_schema": PaperDigest.model_json_schema(),
}

SYSTEM_EXTRACT = """You are an expert research communicator who makes AI papers accessible to software engineers.
Be concrete. Use plain English. Avoid jargon and LaTeX notation in the output fields.
When you see equations, translate them to plain-English descriptions.
Focus on practical usefulness: what can an engineer build or understand from this paper?""".strip()

def extract_digest(meta: dict, pdf_text: str) -> PaperDigest:
    """Call Claude Sonnet to extract structured PaperDigest from paper content."""
    user_content = f"""Paper title: {meta['title']}
Authors: {', '.join(meta['authors'][:5])}
Year: {meta['year']}

Abstract:
{meta['abstract']}

Paper text (first ~18,000 characters):
{pdf_text[:16000]}

Extract the structured digest now using the extract_paper_digest tool.
Be precise about numbers in key_results. Be honest about limitations.
Make one_liner and practitioner_tldr genuinely useful to a practitioner.""".strip()

    resp = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=2048,
        system=SYSTEM_EXTRACT,
        tools=[EXTRACT_TOOL],
        tool_choice={"type": "tool", "name": "extract_paper_digest"},
        messages=[{"role": "user", "content": user_content}],
    )

    tool_block = next(b for b in resp.content if b.type == "tool_use")
    digest = PaperDigest(**tool_block.input)
    # Override title from metadata (more reliable)
    digest.title = meta['title']
    return digest

# ── Run extraction ────────────────────────────────────────────────────────────
print("⏳ Fetching PDF text…")
pdf_text = fetch_pdf_text(DEMO_PAPER)
print(f"   Got {len(pdf_text):,} chars from PDF")

print("⏳ Extracting digest with Claude Sonnet…")
digest = extract_digest(meta, pdf_text)
print("✅ Extraction complete!")


In [ ]:
# ── CodeLayer ────────────────────────────────────────────────────────────────

SYSTEM_CODE = """You are a senior Python engineer who writes minimal, runnable educational code.
Given a paper's core method, write the SMALLEST possible Python snippet that demonstrates
the key idea. Rules:
- Only use stdlib, numpy, or torch (assume numpy always available)
- Max 40 lines of code
- Include a short demo that prints output
- Do NOT explain the code in comments — use a docstring at the top
- The code must run without modification""".strip()

def generate_code_example(digest: PaperDigest) -> str:
    """Use Claude Haiku to generate a minimal runnable code example for the core method."""
    prompt = f"""Paper: {digest.title}

Core method: {digest.method_summary}

One-liner: {digest.one_liner}

Tags: {', '.join(digest.tags)}

Write a minimal Python snippet (≤40 lines) that demonstrates the core idea of this paper.
Return ONLY the Python code block, no prose before or after.""".strip()

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=800,
        system=SYSTEM_CODE,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = resp.content[0].text.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = re.sub(r"^```python\n?|^```\n?|```$", "", raw, flags=re.MULTILINE).strip()
    return raw

print("⏳ Generating code example with Claude Haiku…")
code_example = generate_code_example(digest)
print("✅ Code example generated!")
print()
print("─" * 60)
print(code_example)
print("─" * 60)


In [ ]:
# ── Render the Full Digest ───────────────────────────────────────────────────
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.markdown import Markdown
from rich import box

console = Console()

def render_digest(digest: PaperDigest, code_snippet: str):
    console.print()
    console.print(Panel(f"[bold cyan]{digest.title}[/bold cyan]", title="📄 paper-distiller", expand=True))

    console.print(f"\n[bold green]⚡ One-liner:[/bold green]")
    console.print(f"   {digest.one_liner}")

    console.print(f"\n[bold blue]🔬 Method:[/bold blue]")
    for line in textwrap.wrap(digest.method_summary, width=80):
        console.print(f"   {line}")

    if digest.key_results:
        table = Table(title="📊 Key Results", box=box.SIMPLE, show_header=True)
        table.add_column("Dataset", style="cyan")
        table.add_column("Metric", style="magenta")
        table.add_column("Value", style="bold green")
        table.add_column("vs.", style="dim")
        for r in digest.key_results:
            table.add_row(r.dataset, r.metric, r.value, r.comparison or "")
        console.print(table)

    console.print(f"[bold yellow]📚 Prerequisites:[/bold yellow]")
    for p in digest.prerequisites:
        console.print(f"   • {p}")

    console.print(f"\n[bold red]⚠️  Limitations:[/bold red]")
    for lim in digest.limitations:
        console.print(f"   • {lim}")

    console.print(f"\n[bold]🛠️  Practitioner TL;DR:[/bold]")
    for line in textwrap.wrap(digest.practitioner_tldr, width=80):
        console.print(f"   {line}")

    console.print(f"\n[dim]Tags: {' · '.join('#' + t for t in digest.tags)}[/dim]")

    console.print(f"\n[bold]💻 Code Example:[/bold]")
    console.print(Panel(code_snippet, expand=False, border_style="dim"))

render_digest(digest, code_example)


In [ ]:
# ── Export to JSON + Markdown ────────────────────────────────────────────────

def to_markdown(digest: PaperDigest, code_snippet: str) -> str:
    lines = [
        f"# {digest.title}",
        "",
        f"**⚡ One-liner:** {digest.one_liner}",
        "",
        "## 🔬 Method",
        digest.method_summary,
        "",
    ]
    if digest.key_results:
        lines += ["## 📊 Key Results", "", "| Dataset | Metric | Value | vs. |", "|---------|--------|-------|-----|"]
        for r in digest.key_results:
            lines.append(f"| {r.dataset} | {r.metric} | {r.value} | {r.comparison or ''} |")
        lines.append("")
    lines += [
        "## 📚 Prerequisites",
        *[f"- {p}" for p in digest.prerequisites],
        "",
        "## ⚠️ Limitations",
        *[f"- {lim}" for lim in digest.limitations],
        "",
        "## 🛠️ Practitioner TL;DR",
        digest.practitioner_tldr,
        "",
        "## 💻 Code Example",
        "```python",
        code_snippet,
        "```",
        "",
        f"**Tags:** {' · '.join('#' + t for t in digest.tags)}",
    ]
    return "\n".join(lines)

md_output   = to_markdown(digest, code_example)
json_output = digest.model_dump_json(indent=2)

print("✅ Outputs ready")
print(f"   Markdown: {len(md_output):,} chars")
print(f"   JSON:     {len(json_output):,} chars")

# Save locally in Colab
with open("/content/digest_sample.md", "w") as f: f.write(md_output)
with open("/content/digest_sample.json", "w") as f: f.write(json_output)
print("\n📁 Saved to /content/digest_sample.md and /content/digest_sample.json")


In [ ]:
# ── Project Scaffold Generator ───────────────────────────────────────────────
# This creates the real file structure for the paper-distiller GitHub repo

import pathlib

ROOT = pathlib.Path("/content/paper-distiller")

scaffold = {
    "paper_distiller/__init__.py": '''"""paper-distiller: Turn any arXiv paper into a practitioner-ready digest."""
__version__ = "0.1.0"
from .pipeline import distill
__all__ = ["distill"]
''',
    "paper_distiller/models.py": '''from pydantic import BaseModel, Field
from typing import Optional

class BenchmarkResult(BaseModel):
    dataset: str
    metric: str
    value: str
    comparison: Optional[str] = None

class PaperDigest(BaseModel):
    title: str
    one_liner: str
    method_summary: str
    key_results: list[BenchmarkResult] = Field(default_factory=list)
    prerequisites: list[str] = Field(default_factory=list)
    limitations: list[str] = Field(default_factory=list)
    practitioner_tldr: str
    tags: list[str] = Field(default_factory=list)
    code_example: str = ""
''',
    "paper_distiller/fetch.py": "# FetchLayer: arXiv API + PDF download\n# (implement using code from Lesson 56)\n",
    "paper_distiller/extract.py": "# ExtractLayer: Claude Sonnet tool-forced extraction\n",
    "paper_distiller/codegen.py": "# CodeLayer: Claude Haiku code example generation\n",
    "paper_distiller/pipeline.py": '''from .fetch import fetch_metadata, fetch_pdf_text
from .extract import extract_digest
from .codegen import generate_code_example
from .models import PaperDigest

async def distill(url_or_id: str) -> PaperDigest:
    """Full pipeline: arXiv URL/ID → PaperDigest."""
    from .fetch import parse_arxiv_id
    paper_id = parse_arxiv_id(url_or_id)
    meta = fetch_metadata(paper_id)
    text = fetch_pdf_text(paper_id)
    digest = extract_digest(meta, text)
    digest.code_example = generate_code_example(digest)
    return digest
''',
    "paper_distiller/cli.py": "# CLI entrypoint (typer, added in L60)\n",
    "tests/__init__.py": "",
    "tests/test_models.py": '''from paper_distiller.models import PaperDigest, BenchmarkResult

def test_paper_digest_minimal():
    d = PaperDigest(title="T", one_liner="OL", method_summary="MS", practitioner_tldr="PT")
    assert d.key_results == []

def test_benchmark_result():
    r = BenchmarkResult(dataset="SQuAD", metric="F1", value="90.2%")
    assert r.comparison is None
''',
    "tests/conftest.py": "# Pytest fixtures (mock Claude client for unit tests)\n",
    "evals/golden_papers.json": "[]  # Add (arxiv_id, expected_one_liner) pairs\n",
    ".github/workflows/ci.yml": '''name: CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -e ".[dev]"
      - run: pytest tests/ -v
''',
    "pyproject.toml": '''[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "paper-distiller"
version = "0.1.0"
description = "Turn any arXiv paper into a practitioner-ready digest using Claude"
readme = "README.md"
license = { text = "MIT" }
requires-python = ">=3.10"
dependencies = [
    "anthropic>=0.40",
    "pdfplumber>=0.11",
    "pydantic>=2.0",
    "requests>=2.31",
    "rich>=13.0",
]

[project.optional-dependencies]
dev = ["pytest>=8.0", "pytest-asyncio>=0.23"]
cli = ["typer>=0.12"]

[project.scripts]
distill = "paper_distiller.cli:app"

[project.urls]
Homepage = "https://github.com/YOUR_GITHUB/paper-distiller"
''',
    "README.md": '''# paper-distiller 📄➡️🔬

> Turn any arXiv paper into a practitioner-ready digest in seconds.

```bash
pip install paper-distiller
distill 1706.03762
```

## What you get

- **⚡ One-liner** — the core contribution in plain English
- **🔬 Method** — how it works, no equations
- **📊 Results** — key numbers, what they beat
- **📚 Prerequisites** — what to read first
- **⚠️ Limitations** — honest caveats
- **🛠️ TL;DR** — what YOU should do with this
- **💻 Code** — minimal runnable implementation

## Powered by Claude

Uses `claude-sonnet-4-5` for extraction and `claude-haiku-4-5` for code generation.
Requires `ANTHROPIC_API_KEY`.
''',
    ".gitignore": "__pycache__/\n*.egg-info/\ndist/\n.env\n*.pyc\n",
    "LICENSE": "MIT License\n\nCopyright (c) 2026 Gourav Khanijoe\n",
}

# Write all files
for rel_path, content in scaffold.items():
    fpath = ROOT / rel_path
    fpath.parent.mkdir(parents=True, exist_ok=True)
    fpath.write_text(content)

# Print tree
print("📁 paper-distiller/ scaffold:")
for p in sorted(ROOT.rglob("*")):
    if p.is_file():
        rel = p.relative_to(ROOT)
        print(f"   {rel}")

print(f"\n✅ {len(scaffold)} files created under /content/paper-distiller/")
print("\nNext steps:")
print("  1. cd /content/paper-distiller && git init && git add . && git commit -m 'feat: initial scaffold'")
print("  2. Create repo on GitHub: paper-distiller")
print("  3. git remote add origin git@github.com:YOUR_GITHUB/paper-distiller.git && git push -u origin main")


## 💡 EXPERIMENTS — Try These

1. **Different paper:** Replace `DEMO_PAPER = "1706.03762"` with any arXiv ID you're curious about. Try `2005.14165` (GPT-3) or `2302.13971` (LLaMA).

2. **Cheaper extraction:** Swap `claude-sonnet-4-5` → `claude-haiku-4-5-20251001` in `extract_digest()`. How much quality drops on complex papers?

3. **Multi-paper comparison:** Loop over 3 papers and compare their `one_liner` fields side by side. Useful for "which paper should I read first?"

4. **Add a scoring field:** Add `novelty_score: int = Field(ge=1, le=5)` to `PaperDigest` and prompt Claude to rate how novel the contribution is vs. prior work.

5. **Eval golden set:** In `evals/golden_papers.json`, add 3 papers you know well. Write an assertion that checks `digest.tags` contains the right domain.


## ⚠️ 10 Pitfalls — paper-distiller Edition

| # | Pitfall | Fix |
|---|---------|-----|
| 1 | **Using PDF URL expiry** | Cache downloaded PDFs; arXiv URLs are stable but rate-limit aggressively |
| 2 | **Truncating intro only** | Papers hide key contributions in Section 3–4; always include methods section text |
| 3 | **Hallucinated benchmark numbers** | Always extract numbers with `key_results[]`; never free-form; validate value is numeric string |
| 4 | **one_liner over 30 words** | Add word count check in Pydantic validator; re-prompt if exceeded |
| 5 | **Code example that doesn't run** | Add `exec()` guard in tests; if it raises SyntaxError, trigger a re-generation |
| 6 | **Sonnet for ALL papers** | Use Haiku for short abstracts-only mode; Sonnet only when full PDF text is needed |
| 7 | **No rate limit on arXiv fetches** | arXiv asks for max 3 req/sec; add `time.sleep(0.5)` between batch calls |
| 8 | **Ignoring scanned PDFs** | pdfplumber returns empty string on scanned PDFs; fall back to abstract-only mode |
| 9 | **tags too generic** | Reject tags like "machine learning" or "AI"; require at least one specific architecture/task tag |
| 10 | **No eval harness before v1.0** | Write 5 golden tests before any public release; accuracy regressions sink OSS trust fast |


## 📝 Homework (before Lesson 57)

1. **Run the full pipeline on 5 papers** you care about (any domain). Check if the one-liner captures what you'd say about the paper.

2. **Initialize a real GitHub repo** named `paper-distiller` using the scaffold from Cell 12. Add it to your GitHub profile.

3. **Write 3 unit tests** in `tests/test_models.py` — test that `PaperDigest` raises a validation error if `one_liner` has more than 30 words (add a validator), and that `BenchmarkResult.value` is non-empty.

4. **Cost tracking:** Wrap `extract_digest()` and `generate_code_example()` with the `CostMeter` pattern from Phase 5 (L51). Log `total_cost_usd` per paper. What's the average cost?

5. **Star 3 similar projects** on GitHub (`arxiv-sanity`, `paperswithcode`, `elicit`). Read their README and Issues tabs. Write down: what gap does paper-distiller fill that these don't?

---

## Next: Lesson 57 — paper-distiller Core Pipeline

In L57 you'll:
- Handle scanned PDFs gracefully (abstract-only fallback)
- Add section detection (identify Methods / Results / Conclusion sections by text patterns)
- Build a batch distiller for processing 10+ papers overnight
- Add the golden eval harness with LLM-judge scoring
- Write `distill()` as a proper async function with cost tracking

**Your OSS journey has begun. Ship early, ship often.** 🚀
